## 1. Data loading and project paths
Load market and trade CSVs and inspect dataset dimensions.


In [ ]:
## 1. Load datasets
from pathlib import Path

import pandas as pd

# Set the project root relative to this notebook's location.
PROJECT_ROOT = Path("../../")

# Define the directory containing the raw extension dataset.
DATA_DIR = PROJECT_ROOT / "data" / "Extension Data" / "Raw Data"

# Define the paths to the market and trade CSV files.
MARKETS_PATH = DATA_DIR / "markets.csv"
TRADES_PATH = DATA_DIR / "trades.csv"

# Load the market-level and trade-level datasets into pandas DataFrames.
markets = pd.read_csv(MARKETS_PATH)
trades = pd.read_csv(TRADES_PATH)

# Display the number of rows and columns in each dataset.
print(f"Markets shape: {markets.shape}")
print(f"Trades shape: {trades.shape}")


## 2. Market metadata and category distributions
Inspect missing values and distributions of market type, status, and result.


In [ ]:
import matplotlib.pyplot as plt

# Count categories and missing values.
category_columns = ["market_type", "status", "result"]

display(pd.DataFrame({
    "Distinct categories": markets[category_columns].nunique(),
    "Missing rows": markets[category_columns].isna().sum(),
}))

# Show the number of markets in each category.
for column in category_columns:
    print(f"\n{column}")
    display(
        markets[column]
        .fillna("Missing")
        .value_counts()
        .rename("Markets")
        .to_frame()
    )



## 3. Market-series concentration
Use ticker prefixes to identify market series and visualize the largest groups.


In [ ]:
# There is no topic-category column, so use ticker prefixes as series groups.
# These are market series, not broad topics such as sports or politics.
market_series = markets["ticker"].str.split("-", n=1).str[0]
series_counts = market_series.value_counts()

print(f"\nDistinct market series: {market_series.nunique():,}")

# Display the 10 largest series.
top_series = series_counts.head(10).rename("Markets").to_frame()
top_series["Share of all markets (%)"] = (
    top_series["Markets"] / len(markets) * 100
).round(2)

display(top_series)

# Plot the 10 largest series.
top_series["Markets"].sort_values().plot.barh(figsize=(9, 4))
plt.title("Top 10 market series by number of markets")
plt.xlabel("Number of markets")
plt.ylabel("Market series")
plt.tight_layout()
plt.show()

# Simple conclusions about concentration.
print(f"Largest series: {series_counts.iloc[0] / len(markets):.1%} of markets")
print(f"Top 10 series combined: {series_counts.head(10).sum() / len(markets):.1%} of markets")


## 4. Human-readable series labels
Define descriptive labels for selected ticker prefixes and common sports series.


In [ ]:
# Add or adjust names here as we review more series.
series_names = {
    "KXBTC": "Bitcoin — price range",
    "KXBTCD": "Bitcoin — price threshold",
    "KXETH": "Ethereum — price range",
    "KXETHD": "Ethereum — price threshold",
    "KXDOGE": "Dogecoin — price range",
    "KXDOGED": "Dogecoin — price threshold",
    "KXXRPD": "XRP (Ripple) — price threshold",
    "KXSHIBA": "Shiba Inu — price range",
    "KXSHIBAD": "Shiba Inu — price threshold",
    "KXNASDAQ100": "Nasdaq-100 — price level",
    "KXNASDAQ100U": "Nasdaq-100 — above a price level",
    "KXINX": "S&P 500 — price level",
    "KXINXU": "S&P 500 — above a price level",
    "KXMVENFLMULTIGAMEEXTENDED": "NFL — multi-game combinations (extended)",
    "KXMVENFLMULTIGAME": "NFL — multi-game combinations",
    "KXMVENFLSINGLEGAME": "NFL — single-game combinations",
    "KXMVENBASINGLEGAME": "NBA — single-game combinations",
    "KXMVESPORTSMULTIGAMEEXTENDED": "Sports — multi-game combinations (extended)",
    "KXNFLANYTD": "NFL — anytime touchdown scorer",
    "KXNFLFIRSTTD": "NFL — first touchdown scorer",
    "KXNFL2TD": "NFL — two or more touchdowns",
    "KXNFLRECYDS": "NFL — receiving yards",
    "KXNFLRSHYDS": "NFL — rushing yards",
    "KXNFLPASSYDS": "NFL — passing yards",
    "KXNFLREC": "NFL — receptions",
    "KXNFLWINS": "NFL — season wins",
    "KXPGATOUR": "Golf — PGA Tour winner",
    "KXATPMATCH": "Tennis — ATP match winner",
    "KXWTAMATCH": "Tennis — WTA match winner",
    "KXHIGHDEN": "Weather — Denver high temperature",
    "KXHIGHAUS": "Weather — Austin high temperature",
    "KXHIGHCHI": "Weather — Chicago high temperature",
    "KXHIGHNY": "Weather — New York City high temperature",
    "KXHIGHPHIL": "Weather — Philadelphia high temperature",
    "KXHIGHMIA": "Weather — Miami high temperature",
    "KXHIGHLAX": "Weather — Los Angeles high temperature",
    "KXTRUMPMENTION": "Donald Trump — words mentioned",
    "KXTRUMPMENTIONB": "Donald Trump — words mentioned (series B)",
    "KXSECPRESSMENTION": "White House press secretary — words mentioned",
    "KXSPOTIFYD": "Spotify — top US song",
    "KXSPOTIFY2D": "Spotify — top US song (alternate series)",
    "KXSPOTIFYGLOBALD": "Spotify — top global song",
    "KXAPPRANKFREE": "Apps — top free US iPhone app",
    "KXAPPRANKFREE2": "Apps — second-ranked free US iPhone app",
    "KXOSCARWINNERS": "Oscars — winner combinations",
    "KXMARMAD": "College basketball — national champion",
}

# Expand common sports patterns.
leagues = {
    "NFL": "NFL",
    "NBA": "NBA",
    "WNBA": "WNBA",
    "MLB": "MLB",
    "NHL": "NHL",
    "MLS": "MLS",
    "NCAAF": "College football",
    "NCAAMB": "Men's college basketball",
}

for league, name in leagues.items():
    descriptions = {
        "GAME": "game winner",
        "SPREAD": "point spread",
        "TOTAL": "total points",
    }

    if league == "MLB":
        descriptions.update(SPREAD="run spread", TOTAL="total runs")
    elif league in ["NHL", "MLS"]:
        descriptions.update(SPREAD="goal spread", TOTAL="total goals")

    for suffix, description in descriptions.items():
        series_names[f"KX{league}{suffix}"] = f"{name} — {description}"



## 5. Series lookup and coverage
Create a series-level reference table, evaluate label coverage, and export the lookup CSV.


In [ ]:
# Create one row for every ticker prefix.
series_lookup = (
    markets.assign(series=markets["ticker"].str.split("-", n=1).str[0])
    .groupby("series")
    .agg(
        markets=("ticker", "size"),
        example_title=("title", "first"),
    )
    .sort_values("markets", ascending=False)
    .reset_index()
)

series_lookup["readable_name"] = series_lookup["series"].map(series_names)

series_lookup["label_status"] = (
    series_lookup["readable_name"]
    .notna()
    .map({True: "Inferred label", False: "Needs review"})
)

# Use an example market title when a series has no assigned name.
series_lookup["readable_name"] = series_lookup["readable_name"].fillna(
    series_lookup["example_title"]
)

series_lookup["share_pct"] = (
    series_lookup["markets"] / len(markets) * 100
).round(2)

series_lookup = series_lookup[
    [
        "series",
        "readable_name",
        "label_status",
        "markets",
        "share_pct",
        "example_title",
    ]
]

display(series_lookup.head(30))

labeled = series_lookup["label_status"].eq("Inferred label")

print(f"Named series: {labeled.sum():,} of {len(series_lookup):,}")
print(
    f"Share of market rows with a name: "
    f"{series_lookup.loc[labeled, 'markets'].sum() / len(markets):.1%}"
)

# Save all series for review.
lookup_path = PROJECT_ROOT / "output" / "eda" / "market_series_names.csv"
lookup_path.parent.mkdir(parents=True, exist_ok=True)
series_lookup.to_csv(lookup_path, index=False)

print(f"Full lookup saved to: {lookup_path}")


## 6. Category summaries and visualization
Apply the inferred labels to markets and inspect the resulting category distribution.


In [ ]:
import matplotlib.pyplot as plt

# Append readable categories; keep unnamed series as their original ticker.
markets["series"] = markets["ticker"].str.split("-", n=1).str[0]
markets["category"] = (
    markets["series"]
    .map(series_names)
    .fillna(markets["series"])
    .fillna("Unknown")
)

# Count markets per category.
category_counts = markets["category"].value_counts()

category_summary = category_counts.rename("Markets").to_frame()
category_summary["Share (%)"] = (
    category_summary["Markets"] / len(markets) * 100
).round(2)

display(category_summary.head(10))

# Regenerate the figure with readable labels.
ax = category_counts.head(10).sort_values().plot.barh(
    figsize=(11, 5),
    color="steelblue",
)

ax.set_title("Top 10 market categories by number of markets")
ax.set_xlabel("Number of markets")
ax.set_ylabel("Category (inferred from market series)")

plt.tight_layout()
plt.show()

print(f"Distinct categories: {category_counts.size:,}")
print(f"Largest category: {category_counts.iloc[0] / len(markets):.1%} of markets")
print(f"Top 10 combined: {category_counts.head(10).sum() / len(markets):.1%} of markets")
markets.loc[
    ~markets["series"].isin(series_names),
    ["series", "title"],
].drop_duplicates("series")


## 7. Maker and taker gross excess returns
Match trades to finalized YES/NO outcomes and aggregate gross excess returns in chunks.


In [ ]:
resolved = markets.loc[
    markets["status"].eq("finalized") & markets["result"].isin(["yes", "no"])
].set_index("ticker")["result"]

maker_total = taker_total = n = 0

for t in pd.read_csv(
    TRADES_PATH,
    usecols=["ticker", "taker_side", "yes_price", "no_price"],
    chunksize=500_000,
):
    t["result"] = t["ticker"].map(resolved)
    t = t[t["result"].notna() & t["taker_side"].isin(["yes", "no"])]
    yes = t["taker_side"].eq("yes")
    won = t["taker_side"].eq(t["result"]).astype(float)

    taker_total += (won - t["yes_price"].where(yes, t["no_price"]) / 100).sum()
    maker_total += (1 - won - t["no_price"].where(yes, t["yes_price"]) / 100).sum()
    n += len(t)

if n:
    print(f"Maker gross excess return: {maker_total / n:+.2%}")
    print(f"Taker gross excess return: {taker_total / n:+.2%}")
    print(f"Matched resolved trades: {n:,}")
else:
    print("No trades matched finalized markets.")


## 8. Prediction prototype: dependencies and parameters
Import modeling libraries and set the lookback window, cluster count, and minimum cluster size.


In [ ]:
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

HOURS, CLUSTERS, MIN_CLUSTER = 72, 8, 30



## 9. Eligible contracts and pre-cutoff trade history
Filter finalized contracts and construct historical features using only trades before each cutoff.


In [ ]:
m = markets.loc[
    markets["status"].eq("finalized") & markets["result"].isin(["yes", "no"])
].copy()

assert m["ticker"].is_unique, "Market tickers must be unique."
m["close"] = pd.to_datetime(m["close_time"], utc=True, format="mixed")
m["cutoff"] = m["close"] - pd.Timedelta(hours=HOURS)
m["opened"] = pd.to_datetime(m["open_time"], utc=True, format="mixed")
m = m.loc[m["opened"].le(m["cutoff"])].dropna(subset=["event_ticker", "title"])
pieces = []
cutoffs = m.set_index("ticker")["cutoff"]

for t in pd.read_csv(
    TRADES_PATH,
    usecols=["ticker", "created_time", "yes_price", "count"],
    chunksize=500_000,
):
    t["time"] = pd.to_datetime(t["created_time"], utc=True, format="mixed")
    t = t.loc[t["time"].le(t["ticker"].map(cutoffs))].sort_values("time")
    pieces.append(t.groupby("ticker").agg(
        price=("yes_price", "last"), last_trade=("time", "last"),
        volume=("count", "sum"), trades=("count", "size"),
    ))

history = pd.concat(pieces).sort_values("last_trade").groupby(level=0).agg(
    price=("price", "last"), last_trade=("last_trade", "last"),
    volume=("volume", "sum"), trades=("trades", "sum"),
)
history = history.rename(columns={"volume": "pre_cutoff_volume"})
d = m.join(history, on="ticker", how="inner").reset_index(drop=True)



## 10. Target construction and numerical features
Compute realized deviations from observed prices and summarize historical trading activity.


In [ ]:
d["price"] /= 100
d["target"] = d["result"].eq("yes").astype(float) - d["price"]
d["age_days"] = (d["cutoff"] - d["opened"]).dt.total_seconds() / 86400
d["stale_days"] = (d["cutoff"] - d["last_trade"]).dt.total_seconds() / 86400
d["log_volume"] = np.log1p(d["pre_cutoff_volume"])
d["log_trades"] = np.log1p(d["trades"])

print(f"Eligible contracts with pre-cutoff trades: {len(d):,}")


## 11. Event-level temporal train/test split
Separate events chronologically and exclude training events that close after the test period begins.


In [ ]:
events = d.groupby("event_ticker").agg(close=("close", "max"), cutoff=("cutoff", "min"))
events = events.sort_values("cutoff")
assert len(events) >= 10, "Too few eligible events for this prototype."

test_events = events.index[int(len(events) * 0.8):]
test_start = events.loc[test_events, "cutoff"].min()
train_events = events.index[events["close"].lt(test_start)]

train = d["event_ticker"].isin(train_events).to_numpy()
test = d["event_ticker"].isin(test_events).to_numpy()
assert train.sum() >= MIN_CLUSTER and test.sum(), "Insufficient data after temporal filtering."

print(f"Train: {train.sum():,} contracts | Test: {test.sum():,} contracts")


## 12. Text and numeric feature engineering
Construct wording features, standardized numeric variables, and reduced TF-IDF text representations.


In [ ]:
text = d["title"].fillna("")
d["wording_score"] = (
    text.str.count(r"(?i)\b(win|wins|gain|gains|improve)\b")
    - text.str.count(r"(?i)\b(lose|loses|loss|decline|fail)\b")
)

features = ["price", "age_days", "stale_days", "log_volume", "log_trades", "wording_score"]
scaler = StandardScaler().fit(d.loc[train, features])
numeric = scaler.transform(d[features])

tfidf = TfidfVectorizer(stop_words="english", max_features=5000, min_df=2)
text_train = tfidf.fit_transform(text[train])
svd = TruncatedSVD(n_components=min(32, min(text_train.shape) - 1), random_state=42)
svd.fit(text_train)
semantic = svd.transform(tfidf.transform(text))



## 13. Contract clustering
Fit MiniBatchKMeans on training contracts and assign clusters to the full eligible dataset.


In [ ]:
X = np.column_stack([semantic, numeric])
clustering = MiniBatchKMeans(
    n_clusters=min(CLUSTERS, train.sum()), random_state=42, n_init=10
).fit(X[train])
d["cluster"] = clustering.predict(X)


## 14. Cluster visualization and examples
Project contracts into two dimensions and inspect cluster sizes and representative market titles.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Fit the projection on training data and display a sample of contracts.
projection = PCA(n_components=2).fit(X[train])
sample = d.sample(min(5000, len(d)), random_state=42)
points = projection.transform(X[sample.index])

plt.figure(figsize=(9, 5))
scatter = plt.scatter(
    points[:, 0], points[:, 1],
    c=sample["cluster"], cmap="tab10", s=10, alpha=0.5,
)
plt.legend(*scatter.legend_elements(), title="Cluster", bbox_to_anchor=(1.02, 1))
plt.title("Contract clusters — sample of up to 5,000 contracts")
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.tight_layout()
plt.show()

# Show all cluster sizes and three distinct example titles per cluster.
summary = d.groupby("cluster").agg(
    contracts=("ticker", "size"),
    example_titles=("title", lambda s: " | ".join(s.drop_duplicates().head(3))),
)

with pd.option_context("display.max_colwidth", 180):
    display(summary)


## 15. Global and cluster-specific predictive models
Fit a global Ridge baseline and separate cluster models with historical-mean fallback.


In [ ]:
global_model = Ridge(alpha=10).fit(X[train], d.loc[train, "target"])
d["global_prediction"] = global_model.predict(X)
d["prediction"], d["historical_mean"] = np.nan, np.nan

for cluster in sorted(d["cluster"].unique()):
    fit = train & d["cluster"].eq(cluster).to_numpy()
    predict = test & d["cluster"].eq(cluster).to_numpy()
    historical = d.loc[fit, "target"].mean() if fit.any() else d.loc[train, "target"].mean()
    d.loc[predict, ["historical_mean", "prediction"]] = historical

    if fit.sum() >= MIN_CLUSTER:
        model = Ridge(alpha=10).fit(X[fit], d.loc[fit, "target"] - historical)
        if predict.any():
            d.loc[predict, "prediction"] = historical + model.predict(X[predict])



## 16. Probability bounds
Clip implied YES probabilities to the interval [0, 1] and convert them back to deviations.


In [ ]:
# A predicted YES deviation implies a YES probability of price + deviation.
for column in ["prediction", "global_prediction", "historical_mean"]:
    d[column] = (d["price"] + d[column]).clip(0, 1) - d["price"]


## 17. Held-out evaluation
Compare baseline and predictive methods using event-averaged MAE and RMSE, then inspect results by cluster.


In [ ]:
heldout = d.loc[test].copy()
heldout["zero_prediction"] = 0
scores = {}

for name, column in {
    "No deviation": "zero_prediction",
    "Historical cluster mean": "historical_mean",
    "Global model": "global_prediction",
    "Separate cluster models": "prediction",
}.items():
    error = heldout[column] - heldout["target"]
    event_errors = pd.DataFrame({
        "event": heldout["event_ticker"], "absolute": error.abs(), "squared": error**2,
    }).groupby("event").mean()
    scores[name] = {
        "MAE (pp)": event_errors["absolute"].mean() * 100,
        "RMSE (pp)": np.sqrt(event_errors["squared"].mean()) * 100,
    }

display(pd.DataFrame(scores).T.round(2))
display(heldout.groupby("cluster").agg(
    contracts=("ticker", "size"),
    predicted_deviation=("prediction", "mean"),
    realized_deviation=("target", "mean"),
).round(4))